# Ising pseudo-likelihood fit for DGL FraudYelpDataset

We fit the three-relation model
$$
\mathbb P_{\beta,\gamma}(\sigma\mid X,G)
\propto
\exp\left\{
\sum_{r=1}^3 \beta_r \sum_{(i,j)\in E_r}\sigma_i\sigma_j
+2\sum_{i=1}^n \sigma_i X_i^\top\gamma
\right\},
\qquad \sigma_i\in\{-1,1\}.
$$

The full likelihood is not used, because the partition function is intractable on this graph. Instead, this notebook fits the maximum pseudo-likelihood estimator (MPLE). For node \(i\), define
$$
S_{ir}=\sum_{j:(j,i)\in E_r}\sigma_j .
$$
Then
$$
\mathbb P(\sigma_i=1\mid \sigma_{-i},X,G)
=
\operatorname{logit}^{-1}\left(
2\left[
\sum_{r=1}^3 \beta_r S_{ir}+2X_i^\top\gamma
\right]
\right).
$$

The DGL Fraud Yelp graph has three relations. In this notebook I use
$$
\beta_1=\beta_{\text{R-U-R}},\qquad
\beta_2=\beta_{\text{R-S-R}},\qquad
\beta_3=\beta_{\text{R-T-R}}.
$$


In [1]:
# If needed, install dependencies first. Pick the DGL wheel matching your PyTorch version.
%pip install -q scikit-learn pandas
%pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 48.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 46.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 42.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 11.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 4.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━

In [2]:
import math
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

import dgl
from dgl.data import FraudYelpDataset


SEED = 717
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


device(type='cpu')

## Load Fraud Yelp

DGL stores this as a one-node-type heterograph with relation names `net_rur`, `net_rsr`, and `net_rtr`. The original labels are \(0/1\); we convert them to \(\{-1,1\}\) by
$$
\sigma_i = 2y_i-1.
$$
Thus label \(1\) is treated as the positive fraud/spam class.


In [3]:
dataset = FraudYelpDataset(
    random_seed=SEED,
    train_size=0.7,
    val_size=0.1,
    verbose=True,
)

g = dataset[0]
print(g)
print("node types:", g.ntypes)
print("edge types:", g.etypes)
print("canonical edge types:", g.canonical_etypes)


/root/.dgl/yelp.zip:   0%|          | 0.00/18.0M [00:00<?, ?B/s]

Extracting file to /root/.dgl/yelp_a7a80596
Done saving data into cached files.
Graph(num_nodes={'review': 45954},
      num_edges={('review', 'net_rsr', 'review'): 6805486, ('review', 'net_rtr', 'review'): 1147232, ('review', 'net_rur', 'review'): 98630},
      metagraph=[('review', 'review', 'net_rsr'), ('review', 'review', 'net_rtr'), ('review', 'review', 'net_rur')])
node types: ['review']
edge types: ['net_rsr', 'net_rtr', 'net_rur']
canonical edge types: [('review', 'net_rsr', 'review'), ('review', 'net_rtr', 'review'), ('review', 'net_rur', 'review')]


In [4]:
def get_node_data(g, key, ntype=None):
    """Return node data tensor for a graph with one node type or a heterograph."""
    if ntype is None:
        if len(g.ntypes) != 1:
            raise ValueError("Pass ntype explicitly for a graph with multiple node types.")
        ntype = g.ntypes[0]
    try:
        value = g.ndata[key]
        if isinstance(value, dict):
            return value[ntype]
        return value
    except Exception:
        return g.nodes[ntype].data[key]


ntype = g.ntypes[0]

X_raw = get_node_data(g, "feature", ntype).float()
y01 = get_node_data(g, "label", ntype).long()
train_mask = get_node_data(g, "train_mask", ntype).bool()
val_mask = get_node_data(g, "val_mask", ntype).bool()
test_mask = get_node_data(g, "test_mask", ntype).bool()

sigma_all = 2.0 * y01.float() - 1.0

summary = pd.DataFrame({
    "split": ["train", "validation", "test", "all"],
    "n": [
        int(train_mask.sum()),
        int(val_mask.sum()),
        int(test_mask.sum()),
        int(y01.numel()),
    ],
    "positives": [
        int(y01[train_mask].sum()),
        int(y01[val_mask].sum()),
        int(y01[test_mask].sum()),
        int(y01.sum()),
    ],
})
summary["positive_rate"] = summary["positives"] / summary["n"]
summary


,split,n,positives,positive_rate
0,train,32167,4726,0.146921
1,validation,4595,651,0.141676
2,test,9192,1300,0.141427
3,all,45954,6677,0.145297


## Relation order

The estimator below uses the following order:
$$
(\beta_1,\beta_2,\beta_3)
=
(\beta_{\text{R-U-R}},\beta_{\text{R-S-R}},\beta_{\text{R-T-R}}).
$$

In DGL these are `net_rur`, `net_rsr`, and `net_rtr`.


In [5]:
EDGE_TYPES = ["net_rur", "net_rsr", "net_rtr"]
BETA_LABELS = [
    "beta_1: R-U-R / net_rur / same user",
    "beta_2: R-S-R / net_rsr / same product and same star",
    "beta_3: R-T-R / net_rtr / same product and same month",
]

missing = [e for e in EDGE_TYPES if e not in g.etypes]
if missing:
    raise ValueError(f"Missing expected edge types: {missing}. Present edge types: {g.etypes}")

for etype in EDGE_TYPES:
    print(f"{etype:8s}: {g.num_edges(etype=etype):,} directed edges")


net_rur : 98,630 directed edges
net_rsr : 6,805,486 directed edges
net_rtr : 1,147,232 directed edges


## Feature preprocessing

We standardize the 32 node features using only the training nodes and add an intercept column. This is just a reparametrization of the external-field term \(X_i^\top\gamma\). The printed `gamma_raw_scale` below converts the fitted coefficient back to the original feature scale.


In [6]:
def standardize_with_train_stats(X, train_mask, add_intercept=True, eps=1e-12):
    X = X.float()
    mean = X[train_mask].mean(dim=0)
    std = X[train_mask].std(dim=0, unbiased=False).clamp_min(eps)
    X_std = (X - mean) / std
    if add_intercept:
        ones = torch.ones((X.shape[0], 1), dtype=X.dtype, device=X.device)
        X_aug = torch.cat([ones, X_std], dim=1)
    else:
        X_aug = X_std
    return X_aug, mean, std


X_aug, X_mean, X_std = standardize_with_train_stats(X_raw, train_mask, add_intercept=True)
print("X_raw shape:", tuple(X_raw.shape))
print("X_aug shape:", tuple(X_aug.shape))


X_raw shape: (45954, 32)
X_aug shape: (45954, 33)


## Build the local neighbor sums \(S_{ir}\)

For a fair supervised train/validation/test evaluation, validation and test labels should not be used in the local sums. Therefore the default below uses only training labels as observed spins; neighbors outside the training set contribute \(0\).

If you instead want the fully observed MPLE diagnostic, replace `SOURCE_LABEL_MASK_FOR_NEIGHBOR_SUMS = train_mask` by all `True`. That uses validation and test labels inside the covariates and is therefore not a fair node-classification evaluation.


In [17]:
SOURCE_LABEL_MASK_FOR_NEIGHBOR_SUMS = train_mask
# SOURCE_LABEL_MASK_FOR_NEIGHBOR_SUMS = torch.ones_like(train_mask, dtype=torch.bool)  # oracle diagnostic only

# To match the model exactly, use "none".
# If the optimization saturates because of very large degrees, try "degree";
# that fits a degree-normalized variant, not exactly the displayed model.
NEIGHBOR_NORMALIZATION = "degree"  # one of {"none", "degree", "sqrt_degree"}


@torch.no_grad()
def compute_relation_neighbor_sums(g, sigma_source, edge_types, normalize="none", device=None):
    """
    Return S with S[i, r] = sum_{j : (j,i) in E_r} sigma_source[j].

    The DGL fraud graphs are bidirected and have no self-loops, so incoming directed
    edges give the usual undirected-neighbor local field.
    """
    if device is None:
        device = sigma_source.device

    sigma_source = sigma_source.to(device).float()
    n = sigma_source.numel()
    cols = []

    for etype in edge_types:
        src, dst = g.edges(etype=etype)
        src = src.to(device)
        dst = dst.to(device)

        out = torch.zeros(n, dtype=torch.float32, device=device)
        out.index_add_(0, dst, sigma_source[src])

        if normalize in {"degree", "sqrt_degree"}:
            deg = torch.zeros(n, dtype=torch.float32, device=device)
            deg.index_add_(0, dst, torch.ones(dst.shape[0], dtype=torch.float32, device=device))
            if normalize == "degree":
                out = out / deg.clamp_min(1.0)
            else:
                out = out / deg.clamp_min(1.0).sqrt()
        elif normalize == "none":
            pass
        else:
            raise ValueError("normalize must be one of {'none', 'degree', 'sqrt_degree'}")

        cols.append(out)

    return torch.stack(cols, dim=1)


sigma_source = torch.zeros_like(sigma_all)
sigma_source[SOURCE_LABEL_MASK_FOR_NEIGHBOR_SUMS] = sigma_all[SOURCE_LABEL_MASK_FOR_NEIGHBOR_SUMS]

S_raw = compute_relation_neighbor_sums(
    g,
    sigma_source=sigma_source,
    edge_types=EDGE_TYPES,
    normalize=NEIGHBOR_NORMALIZATION,
    device=device,
).cpu()

S_raw_df = pd.DataFrame(S_raw[train_mask].numpy(), columns=EDGE_TYPES).describe().T
S_raw_df


,count,mean,std,min,25%,50%,75%,max
net_rur,32167.0,-0.321973,0.450760,-1.0,-0.750000,0.000000,0.000000,1.0
net_rsr,32167.0,-0.493501,0.163297,-1.0,-0.577982,-0.528689,-0.439697,1.0
net_rtr,32167.0,-0.487702,0.218061,-1.0,-0.613636,-0.500000,-0.388889,1.0


For numerical stability, we divide each \(S_{\cdot r}\) by its training-set standard deviation before optimization. This does not change the model class: after fitting, the notebook converts \(\beta\) back to the original scale.


In [18]:
eps = 1e-12
S_scale = S_raw[train_mask].std(dim=0, unbiased=False).clamp_min(eps)
S_scaled = S_raw / S_scale

pd.DataFrame({
    "edge_type": EDGE_TYPES,
    "S_train_std": S_scale.numpy(),
})


,edge_type,S_train_std
0,net_rur,0.450780
1,net_rsr,0.163295
2,net_rtr,0.218065


## MPLE model

The conditional log-odds used in the binary logistic regression is
$$
\log \frac{\mathbb P(\sigma_i=1\mid\sigma_{-i},X,G)}
{\mathbb P(\sigma_i=-1\mid\sigma_{-i},X,G)}
=
2\left[\sum_{r=1}^3\beta_r S_{ir}+2X_i^\top\gamma\right].
$$


In [19]:
class IsingPseudolikelihood(nn.Module):
    def __init__(self, n_features_aug, n_relations=3):
        super().__init__()
        self.beta = nn.Parameter(torch.zeros(n_relations))
        self.gamma = nn.Parameter(torch.zeros(n_features_aug))

    def logits(self, S_scaled, X_aug):
        # Local field:
        # H_i = sum_r beta_r S_ir + 2 X_i^T gamma.
        H = S_scaled @ self.beta + 2.0 * (X_aug @ self.gamma)
        # log odds P(sigma=+1)/P(sigma=-1) = 2 H_i.
        return 2.0 * H

    def forward(self, S_scaled, X_aug):
        return self.logits(S_scaled, X_aug)


def fit_mple(
    S_scaled,
    X_aug,
    y01,
    train_mask,
    l2_beta=1e-4,
    l2_gamma=1e-4,
    max_iter=200,
    pos_weight=None,
    device=None,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    S_scaled = S_scaled.to(device).float()
    X_aug = X_aug.to(device).float()
    y01 = y01.to(device).float()
    train_mask = train_mask.to(device).bool()

    model = IsingPseudolikelihood(
        n_features_aug=X_aug.shape[1],
        n_relations=S_scaled.shape[1],
    ).to(device)

    if pos_weight is not None:
        pos_weight_tensor = torch.tensor(float(pos_weight), device=device)
    else:
        pos_weight_tensor = None

    optimizer = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=max_iter,
        line_search_fn="strong_wolfe",
    )

    def objective():
        logits = model(S_scaled, X_aug)
        data_loss = F.binary_cross_entropy_with_logits(
            logits[train_mask],
            y01[train_mask],
            pos_weight=pos_weight_tensor,
        )
        penalty = 0.5 * l2_beta * (model.beta @ model.beta)
        # Do not penalize the intercept gamma[0].
        if model.gamma.numel() > 1:
            penalty = penalty + 0.5 * l2_gamma * (model.gamma[1:] @ model.gamma[1:])
        return data_loss + penalty

    def closure():
        optimizer.zero_grad(set_to_none=True)
        loss = objective()
        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        final_loss = objective().item()

    return model, final_loss


# Use the true MPLE by default. Setting USE_POS_WEIGHT=True changes the objective
# and may help classification under imbalance, but it is no longer the ordinary MPLE.
USE_POS_WEIGHT = False

y_train = y01[train_mask]
pos_weight = None
if USE_POS_WEIGHT:
    pos_weight = (y_train.numel() - y_train.sum()).item() / max(float(y_train.sum().item()), 1.0)
    print("Using pos_weight =", pos_weight)

model, final_loss = fit_mple(
    S_scaled=S_scaled,
    X_aug=X_aug,
    y01=y01,
    train_mask=train_mask,
    l2_beta=1e-4,
    l2_gamma=1e-4,
    max_iter=250,
    pos_weight=pos_weight,
    device=device,
)

print("Final regularized training objective:", final_loss)


Final regularized training objective: 0.24803654849529266


In [20]:
@torch.no_grad()
def convert_parameters_to_original_scale(model, S_scale, X_mean, X_std):
    """
    Convert beta back from S_scaled = S_raw / S_scale.

    The gamma conversion uses X_aug = [1, (X_raw - mean)/std].
    """
    beta_scaled = model.beta.detach().cpu()
    gamma_std_scale = model.gamma.detach().cpu()

    beta_original_scale = beta_scaled / S_scale.cpu()

    gamma_raw_scale = torch.empty_like(gamma_std_scale)
    gamma_raw_scale[1:] = gamma_std_scale[1:] / X_std.cpu()
    gamma_raw_scale[0] = gamma_std_scale[0] - (gamma_std_scale[1:] * X_mean.cpu() / X_std.cpu()).sum()

    return beta_scaled, beta_original_scale, gamma_std_scale, gamma_raw_scale


beta_scaled, beta_original_scale, gamma_std_scale, gamma_raw_scale = convert_parameters_to_original_scale(
    model, S_scale, X_mean, X_std
)

beta_df = pd.DataFrame({
    "parameter": BETA_LABELS,
    "estimate_on_scaled_S": beta_scaled.numpy(),
    "estimate_original_S_scale": beta_original_scale.numpy(),
})
beta_df


,parameter,estimate_on_scaled_S,estimate_original_S_scale
0,beta_1: R-U-R / net_rur / same user,1.203031,2.668776
1,beta_2: R-S-R / net_rsr / same product and sam...,0.214759,1.315162
2,beta_3: R-T-R / net_rtr / same product and sam...,0.050431,0.231264


In [21]:
gamma_df = pd.DataFrame({
    "parameter": ["gamma_intercept"] + [f"gamma_feature_{j}" for j in range(X_raw.shape[1])],
    "estimate_standardized_X_scale": gamma_std_scale.numpy(),
    "estimate_raw_X_scale": gamma_raw_scale.numpy(),
})
gamma_df.head(15)


,parameter,estimate_standardized_X_scale,estimate_raw_X_scale
0,gamma_intercept,-0.036160,-4.380577
1,gamma_feature_0,0.076702,0.263906
2,gamma_feature_1,0.009912,0.034309
3,gamma_feature_2,-0.053240,-0.188510
4,gamma_feature_3,-0.046764,-0.234169
5,gamma_feature_4,-0.033633,-0.229728
6,gamma_feature_5,-0.134023,-0.452306
7,gamma_feature_6,0.015455,0.050826
8,gamma_feature_7,-0.026124,-0.090412
9,gamma_feature_8,-0.055497,-0.192427


## Prediction and evaluation

The model gives
$$
\widehat p_i
=
\mathbb P(\sigma_i=1\mid X,G,\text{observed spins})
=
\operatorname{logit}^{-1}\left(
2\left[\sum_r\widehat\beta_r S_{ir}+2X_i^\top\widehat\gamma\right]
\right).
$$

Because this dataset is imbalanced, the threshold \(0.5\) is often not the most useful reporting threshold. The next cells report both threshold \(0.5\) and a threshold selected on validation data to maximize balanced accuracy.


In [22]:
@torch.no_grad()
def predict_prob(model, S_scaled, X_aug, device=None):
    if device is None:
        device = next(model.parameters()).device
    logits = model(S_scaled.to(device).float(), X_aug.to(device).float())
    return torch.sigmoid(logits).detach().cpu(), logits.detach().cpu()


prob, logits = predict_prob(model, S_scaled, X_aug, device=device)
prob[:10]


tensor([0.2430, 0.7209, 0.8482, 0.8062, 0.0970, 0.6924, 0.1080, 0.1324, 0.1314,
        0.1040])

In [23]:
def safe_auc(y_true, score):
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return np.nan
    return roc_auc_score(y_true, score)


def safe_ap(y_true, score):
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return np.nan
    return average_precision_score(y_true, score)


def evaluate_mask(name, mask, y01, prob, threshold=0.5):
    idx = mask.cpu().numpy().astype(bool)
    y_true = y01.cpu().numpy()[idx].astype(int)
    score = prob.cpu().numpy()[idx]
    y_pred = (score >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return {
        "split": name,
        "threshold": threshold,
        "n": int(idx.sum()),
        "positive_rate": float(y_true.mean()),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_pos": precision_score(y_true, y_pred, zero_division=0),
        "recall_pos": recall_score(y_true, y_pred, zero_division=0),
        "f1_pos": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": safe_auc(y_true, score),
        "average_precision": safe_ap(y_true, score),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }


metrics_05 = pd.DataFrame([
    evaluate_mask("train", train_mask, y01, prob, threshold=0.5),
    evaluate_mask("validation", val_mask, y01, prob, threshold=0.5),
    evaluate_mask("test", test_mask, y01, prob, threshold=0.5),
])
metrics_05


,split,threshold,n,positive_rate,accuracy,balanced_accuracy,precision_pos,recall_pos,f1_pos,roc_auc,average_precision,tn,fp,fn,tp
0,train,0.5,32167,0.146921,0.897441,0.708511,0.760306,0.440965,0.558189,0.907930,0.684008,26784,657,2642,2084
1,validation,0.5,4595,0.141676,0.893580,0.691116,0.718919,0.408602,0.521058,0.908763,0.661489,3840,104,385,266
2,test,0.5,9192,0.141427,0.900131,0.705714,0.755348,0.434615,0.551758,0.909424,0.678852,7709,183,735,565


In [24]:
def select_threshold_on_validation(y01, prob, val_mask, objective="balanced_accuracy", n_grid=1001):
    y_true = y01.cpu().numpy()[val_mask.cpu().numpy().astype(bool)].astype(int)
    score = prob.cpu().numpy()[val_mask.cpu().numpy().astype(bool)]

    thresholds = np.linspace(0.0, 1.0, n_grid)
    best_t = 0.5
    best_value = -np.inf

    for t in thresholds:
        pred = (score >= t).astype(int)
        if objective == "balanced_accuracy":
            value = balanced_accuracy_score(y_true, pred)
        elif objective == "f1":
            value = f1_score(y_true, pred, zero_division=0)
        else:
            raise ValueError("objective must be 'balanced_accuracy' or 'f1'")

        if value > best_value:
            best_value = value
            best_t = float(t)

    return best_t, best_value


best_t_balacc, best_val_balacc = select_threshold_on_validation(
    y01, prob, val_mask, objective="balanced_accuracy"
)
best_t_f1, best_val_f1 = select_threshold_on_validation(
    y01, prob, val_mask, objective="f1"
)

print("Best validation threshold for balanced accuracy:", best_t_balacc, "value:", best_val_balacc)
print("Best validation threshold for F1:", best_t_f1, "value:", best_val_f1)


Best validation threshold for balanced accuracy: 0.137 value: 0.8304245224229848
Best validation threshold for F1: 0.26 value: 0.6074766355140186


In [25]:
metrics_val_tuned = pd.DataFrame([
    evaluate_mask("train", train_mask, y01, prob, threshold=best_t_balacc),
    evaluate_mask("validation", val_mask, y01, prob, threshold=best_t_balacc),
    evaluate_mask("test", test_mask, y01, prob, threshold=best_t_balacc),
])
metrics_val_tuned


,split,threshold,n,positive_rate,accuracy,balanced_accuracy,precision_pos,recall_pos,f1_pos,roc_auc,average_precision,tn,fp,fn,tp
0,train,0.137,32167,0.146921,0.791650,0.822185,0.402718,0.865425,0.549657,0.907930,0.684008,21375,6066,636,4090
1,validation,0.137,4595,0.141676,0.795865,0.830425,0.399720,0.878648,0.549472,0.908763,0.661489,3085,859,79,572
2,test,0.137,9192,0.141427,0.797541,0.824590,0.399929,0.862308,0.546429,0.909424,0.678852,6210,1682,179,1121


## Optional: refit with degree-normalized local fields

The displayed model uses the raw sums \(S_{ir}\). On the Yelp graph, relation degrees can be large, so the raw model can have strong saturation. If you want the degree-normalized variant
$$
S_{ir}^{\mathrm{norm}}=\frac{1}{d_{ir}}\sum_{j:(j,i)\in E_r}\sigma_j,
$$
set `NEIGHBOR_NORMALIZATION = "degree"` near the top and rerun the notebook. This is not the same parameterization as the original Gibbs measure, but it is often numerically more stable for node classification.


## Optional: save fitted values


In [26]:
results = {
    "edge_types": EDGE_TYPES,
    "beta_scaled": beta_scaled.numpy().tolist(),
    "beta_original_scale": beta_original_scale.numpy().tolist(),
    "gamma_standardized_X_scale": gamma_std_scale.numpy().tolist(),
    "gamma_raw_X_scale": gamma_raw_scale.numpy().tolist(),
    "neighbor_normalization": NEIGHBOR_NORMALIZATION,
    "source_labels": "train_mask_only" if torch.equal(SOURCE_LABEL_MASK_FOR_NEIGHBOR_SUMS, train_mask) else "custom",
    "threshold_05_metrics": metrics_05.to_dict(orient="records"),
    "threshold_val_balacc": best_t_balacc,
    "threshold_val_balacc_metrics": metrics_val_tuned.to_dict(orient="records"),
}

pd.Series(results)


,0
edge_types,"[net_rur, net_rsr, net_rtr]"
beta_scaled,"[1.2030314207077026, 0.21475932002067566, 0.05..."
beta_original_scale,"[2.668775796890259, 1.3151617050170898, 0.2312..."
gamma_standardized_X_scale,"[-0.03615963086485863, 0.07670202106237411, 0...."
gamma_raw_X_scale,"[-4.3805766105651855, 0.26390552520751953, 0.0..."
neighbor_normalization,degree
source_labels,train_mask_only
threshold_05_metrics,"[{'split': 'train', 'threshold': 0.5, 'n': 321..."
threshold_val_balacc,0.137
threshold_val_balacc_metrics,"[{'split': 'train', 'threshold': 0.137, 'n': 3..."
